In [3]:
# Example 1 - Simple text loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/sample.txt")  # Make sure the file is uploaded
docs = loader.load()

print(docs[0].page_content)  # Show first document's content

RuntimeError: Error loading /content/sample.txt

In [ ]:
# Example 2 - Simple CSV loader
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="/content/data.csv")  # Upload your .csv
docs = loader.load()

print("📄 First Row Content:\n", docs[0].page_content)
print("🗂️ Metadata:\n", docs[0].metadata)


In [ ]:
# Example 3 - Simple PDF loader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/example.pdf")  # Upload your .pdf
pages = loader.load()

print("📄 Page 1 Content:\n", pages[0].page_content[:500])
print("🗂️ Metadata:\n", pages[0].metadata)


In [ ]:
# Example 4 - Simple Web loader
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.databricks.com/aws/en/machine-learning/")
docs = loader.load()

print("📄 Web Page Content:\n", docs[0].page_content[:500])
print("🗂️ Metadata:\n", docs[0].metadata)


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("/content/sample.txt")
docs = loader.load()
print("Original Length:", len(docs[0].page_content))

# Split the data

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

split_docs = splitter.split_documents(docs)

print(f"🔹 Total Chunks: {len(split_docs)}")
print("📄 Sample Chunk:\n", split_docs[0].page_content)

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
os.environ['OLLAMA_API_KEY'] = apiKey

if not apiKey:
    raise RuntimeError(
        "❌ OPENAI_API_KEY not found. Set once with:\n"
        "from google.colab import userdata\n"
        "userdata.set('OPENAI_API_KEY', 'sk-...')"
    )

# 2) Paste your own source text (or press Enter to use defaults)
raw = input(
    "Paste your reference text (or press Enter to use a default cricket sample):\n"
).strip()

if not raw:
    raw = """
Mahendra Singh Dhoni (MSD) captained India and won the 2007 T20 World Cup and the 2011 ODI World Cup.
Sachin Ramesh Tendulkar (SRT) is widely called the 'God of Cricket' for his batting records.
Amitabh Bachchan, nicknamed Big B, is a legendary actor in Indian cinema.
Shah Rukh Khan, often called King Khan, is one of the most popular Bollywood actors worldwide.
Virat Kohli is renowned for batting consistency across formats.
Rohit Sharma is known for explosive batting and captaincy in limited-overs cricket.
"""

# 3) Split into chunks (simple defaults work fine to start)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60)
docs = splitter.create_documents([raw])

# 4) Build vector store + retriever
emb = OllamaEmbeddings(model="nomic-embed-text")   # fast & good
vs = FAISS.from_documents(docs, embedding=emb)
retriever = vs.as_retriever(search_kwargs={"k": 3})

# 5) Define the RAG prompt
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Use ONLY the context to answer.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "If the answer is not in the context, say you don't know.\n"
    "Answer:"
)

# 6) Helper to format retrieved docs
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 7) LLM + chain
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0.2)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 8) Interactive Q&A
print("\n💬 Ask questions about your text. Type 'exit' to quit.")
while True:
    q = input("Q: ").strip()
    if q.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break

    # Optional: peek at retrieved chunks (uncomment to see)
    # retrieved = retriever.get_relevant_documents(q)
    # print("\n[DEBUG] Retrieved context:\n", format_docs(retrieved), "\n")
    print('Asking question to LLM')
    ans = rag_chain.invoke(q)
    print("A:", ans, "\n")


💬 Ask questions about your text. Type 'exit' to quit.
👋 Goodbye!
